# Monitoring part 2: model quality

**Question this notebook answers:** *are the forecasts still good, and what do we do when they are not?*

The data-quality notebook checks the **inputs**. This one checks the **outputs** of the deployed model
(DLinear CPU FP32, chosen in Milestone 2) against the targets (SLOs) set in Milestone 2:

| SLO (Milestone 2) | Target | If violated (M2) |
|---|---|---|
| Point accuracy | rolling 7-day MAE ≤ 0.8 × seasonal-naive MAE | alert; roll back if violated 3 days in a row |
| Interval coverage | rolling 7-day 80% coverage within [75%, 85%] | recalibrate the conformal correction on recent data |
| Inference latency | p95 ≤ 50 ms per forecast | serve last cached forecast |
| Forecast freshness | new forecast ≤ 2 min after each 15-min step | label forecast stale, use fallback |

**Why we can measure real accuracy quickly:** the "true answer" for a forecast is just the occupancy observed
4 hours later. So every forecast can be scored the same day; no long wait for labels.

**How we simulate production:** we replay the saved test-set forecasts **one day at a time**, as if they had just
been scored. Run A uses the real forecasts; run B injects a broken model release to show the rollback.

## 1. Setup and thresholds

In [1]:
import os

import numpy as np
import pandas as pd

OUT = "outputs/monitoring"
os.makedirs(OUT, exist_ok=True)
SITES = ["caltech", "jpl"]
TZ = "America/Los_Angeles"

# Milestone 2 SLOs, plus a WARN level just before each limit
RATIO_WARN, RATIO_SLO = 0.75, 0.80      # MAE(model) / MAE(seasonal naive)
WORSE_THAN_BASELINE = 1.00              # the model is no better than the naive rule
COVERAGE_SLO = (0.75, 0.85)             # 80% interval must cover 75-85% of the truth
COVERAGE_WARN = (0.77, 0.83)
STREAK = 3                              # "3 days in a row" rule from Milestone 2
WINDOW_DAYS = 7                         # rolling window of the SLOs
MIN_DAYS_RATIO = 2                      # judge accuracy after 2 days (catch a bad release fast)
MIN_DAYS_COVERAGE = 7                   # judge coverage on a full week (fewer misses = noisy estimate)

## 2. Load the prediction logs

In production every forecast is logged with its interval and model version. The benchmark already saved exactly
that for the test period: for each forecast, the 16 predicted values (median), the calibrated 80% interval
(lower / upper), and later the actual occupancy. We load the deployed model (**DLinear**) and the baseline
(**seasonal naive**: "same time last week").

Each forecast is assigned to the **day its last value (4 h ahead) is known**: that is the day it can be scored.

In [2]:
def load_log(site, model):
    f = np.load(f"outputs/benchmark_{site}/forecasts/{model}.npz")
    return {
        "y": f["actual"], "pred": f["prediction"], "lower": f["lower"], "upper": f["upper"],
        "origin": f["test_origins"],
        "day": pd.to_datetime(f["forecast_timestamps"][:, -1]).tz_convert(TZ).normalize(),
    }

model = {site: load_log(site, "dlinear_cpu_fp32") for site in SITES}
naive = {site: load_log(site, "seasonal_naive_cpu_numpy") for site in SITES}

for site in SITES:
    assert np.array_equal(model[site]["origin"], naive[site]["origin"])   # same forecasts, same truth
    m = model[site]
    print(f"{site:8s} {len(m['y']):6,} forecasts x 16 steps | {m['day'].nunique()} days | "
          f"MAE {np.abs(m['y'] - m['pred']).mean():.3f} | "
          f"coverage {((m['y'] >= m['lower']) & (m['y'] <= m['upper'])).mean():.1%}")

caltech  12,102 forecasts x 16 steps | 127 days | MAE 0.921 | coverage 76.6%
jpl      12,270 forecasts x 16 steps | 131 days | MAE 1.806 | coverage 79.8%


These overall numbers match the Milestone 2 tables (Caltech MAE 0.921 / coverage 76.6%, JPL 1.806 / 79.8%).

## 3. Proxy checks: available the moment a forecast is made

Some problems are visible **before** the truth arrives, just by looking at the forecast:

* **Impossible values**: occupancy can never be negative.
* **Interval width**: if intervals suddenly get much wider or narrower, something changed.
* **Disagreement with the naive baseline**: a sudden jump means the model is doing something unusual.

In [3]:
rows = {}
for site in SITES:
    m, b = model[site], naive[site]
    clipped_mae = np.abs(m["y"] - np.maximum(m["pred"], 0)).mean()
    rows[site] = {
        "negative median predictions (%)": 100 * (m["pred"] < 0).mean(),
        "negative lower bounds (%)": 100 * (m["lower"] < 0).mean(),
        "mean interval width (cars)": (m["upper"] - m["lower"]).mean(),
        "mean |model - naive| (cars)": np.abs(m["pred"] - b["pred"]).mean(),
        "MAE as deployed": np.abs(m["y"] - m["pred"]).mean(),
        "MAE if predictions clipped at 0": clipped_mae,
    }
pd.DataFrame(rows).round(3)

,caltech,jpl
negative median predictions (%),12.482,17.036
negative lower bounds (%),45.390,49.355
mean interval width (cars),2.862,6.384
mean |model - naive| (cars),1.147,2.945
MAE as deployed,0.921,1.806
MAE if predictions clipped at 0,0.889,1.694


**Finding:** about 1 in 8 median forecasts is **negative** (mostly at night, when the true value is 0). The
monitor flags this immediately, and the fix is one line: clip predictions at 0, which also lowers MAE (table
above). Lower bounds below 0 are harmless for coverage but should be clipped before showing them to users.

### Accuracy by lead time

Milestone 2 reported pooled metrics only. The monitor also tracks them by lead time, because a forecast for
15 minutes ahead and one for 4 hours ahead do not have the same error.

In [4]:
steps = {"15 min": 0, "1 h": 3, "2 h": 7, "4 h": 15}
lead = []
for site in SITES:
    m, b = model[site], naive[site]
    for name, h in steps.items():
        lead.append({"site": site, "lead time": name,
                     "MAE": np.abs(m["y"][:, h] - m["pred"][:, h]).mean(),
                     "MAE / naive": np.abs(m["y"][:, h] - m["pred"][:, h]).mean()
                                    / np.abs(b["y"][:, h] - b["pred"][:, h]).mean(),
                     "coverage": ((m["y"][:, h] >= m["lower"][:, h]) & (m["y"][:, h] <= m["upper"][:, h])).mean()})
pd.DataFrame(lead).pivot(index="lead time", columns="site").reindex(steps).round(3)

MAE        MAE / naive        coverage       
site      caltech    jpl     caltech    jpl  caltech    jpl
lead time                                                  
15 min      0.375  0.902       0.308  0.286    0.677  0.772
1 h         0.764  1.267       0.629  0.400    0.763  0.810
2 h         1.013  1.733       0.832  0.546    0.775  0.797
4 h         1.104  2.590       0.907  0.811    0.791  0.776

## 4. The response policy

Milestone 2 says: *"roll back if the accuracy SLO is violated 3 days in a row"*. Section 6 shows that applied
literally this rule fires repeatedly on the **real** Caltech data, even though DLinear is still clearly better than
the fallback. Rolling back (or falling back to seasonal naive) would then make forecasts **worse**. So the policy
is split into two levels:

| Situation (rolling 7-day, 3 days in a row) | Meaning | Action |
|---|---|---|
| MAE ratio > 0.80 | model misses its **target** | page ML on-call, open a retraining ticket |
| MAE ratio ≥ 1.00 | model is **no better than the naive rule** | automatic **rollback** to the previous version, or **fallback** to seasonal naive if there is none |
| coverage outside [75%, 85%] (any day) | intervals are miscalibrated | automatic **recalibration**: recompute the conformal correction on the last 7 days |

**Recalibration** is the same CQR step as in Milestone 2, just on recent data: for every recent forecast, compute how
far the truth fell outside the interval (`score = max(lower - y, y - upper)`), take the 80% quantile of those scores
for each lead time, and widen (or narrow) the interval by that amount.

**Fair windows:** every time something changes (new version, rollback, recalibration), the 7-day window
restarts, so a new setup is judged only on its own forecasts. Accuracy is judged after 2 days (a bad release must
be caught fast); coverage only after a full week, because an 80% coverage measured on 1-2 days is too noisy and
would trigger recalibration over and over.

In [5]:
def conformal_offset(y, lower, upper, level=0.8):
    # CQR: how much to widen each lead time so that `level` of recent truths fall inside
    scores = np.maximum(lower - y, y - upper)                      # > 0 when the truth is outside
    n = len(scores)
    q = min(1.0, np.ceil((n + 1) * level) / n)
    return np.quantile(scores, q, axis=0)                          # one value per lead time


def status_ratio(r):
    if np.isnan(r):
        return "n/a"
    return "ALERT" if r > RATIO_SLO else "WARN" if r > RATIO_WARN else "OK"


def status_coverage(c):
    if np.isnan(c):
        return "n/a"
    if c < COVERAGE_SLO[0] or c > COVERAGE_SLO[1]:
        return "ALERT"
    return "WARN" if c < COVERAGE_WARN[0] or c > COVERAGE_WARN[1] else "OK"

### The daily replay

Each day: score the forecasts whose truth is now known, update the rolling metrics, then decide the actions for
**tomorrow**. `versions` holds the model versions that exist (normally only `v1`); `releases` says on which day a
new version goes live. With `act=False` the monitor only watches, so we can compare *with* and *without* the policy.

In [6]:
def replay(site, versions, releases=None, act=True):
    releases = releases or {}
    base = naive[site]
    days = base["day"].unique().sort_values()
    live = {"version": "v1", "previous": None}
    offsets = {name: np.zeros(16) for name in versions}            # calibration belongs to its version
    since_version = since_calibration = days[0]
    ratio_streak = worse_streak = 0
    log = []

    for i, day in enumerate(days):
        actions = []
        if day in releases:                                         # a new version goes live today
            live = {"version": releases[day], "previous": live["version"]}
            since_version = since_calibration = day
            actions.append(f"release {releases[day]}")

        # Serve today's forecasts with the live version and its calibration
        today = (base["day"] == day)
        v = versions[live["version"]]
        off = offsets[live["version"]]
        served = {"day": day, "version": live["version"],
                  "abs_err": np.abs(v["y"][today] - v["pred"][today]).sum(),
                  "naive_abs_err": np.abs(base["y"][today] - base["pred"][today]).sum(),
                  "covered": ((v["y"][today] >= v["lower"][today] - off) & (v["y"][today] <= v["upper"][today] + off)).sum(),
                  "width": (v["upper"][today] - v["lower"][today] + 2 * off).sum(),
                  "negative": (v["pred"][today] < 0).sum(),
                  "values": v["y"][today].size}
        log.append(served)
        hist = pd.DataFrame(log)

        # Rolling 7-day SLO metrics, only on forecasts from the current setup
        def window(since, min_days):
            first_day = max(since, day - pd.Timedelta(days=WINDOW_DAYS - 1))
            w = hist[(hist["day"] >= first_day) & (hist["values"] > 0)]
            return w if (day - since).days + 1 >= min_days and len(w) else None
        wv, wc = window(since_version, MIN_DAYS_RATIO), window(since_calibration, MIN_DAYS_COVERAGE)
        ratio = wv["abs_err"].sum() / wv["naive_abs_err"].sum() if wv is not None and wv["naive_abs_err"].sum() > 0 else np.nan
        coverage = wc["covered"].sum() / wc["values"].sum() if wc is not None else np.nan
        served.update(ratio_7d=ratio, coverage_7d=coverage, s_ratio=status_ratio(ratio), s_coverage=status_coverage(coverage),
                      daily_ratio=served["abs_err"] / served["naive_abs_err"] if served["naive_abs_err"] else np.nan,
                      width_mean=served["width"] / max(served["values"], 1),
                      negative_pct=100 * served["negative"] / max(served["values"], 1))

        # Decide tomorrow's actions
        ratio_streak = ratio_streak + 1 if served["s_ratio"] == "ALERT" else 0
        worse_streak = worse_streak + 1 if ratio >= WORSE_THAN_BASELINE else 0
        if act and worse_streak >= STREAK:
            if live["previous"]:
                actions.append(f"ROLLBACK {live['version']} -> {live['previous']}")
                live = {"version": live["previous"], "previous": None}
            else:
                actions.append("FALLBACK to seasonal naive")
            since_version = since_calibration = day + pd.Timedelta(days=1)
            ratio_streak = worse_streak = 0
        elif ratio_streak == STREAK:
            actions.append("PAGE: below target 3 days -> retraining ticket")
        if act and served["s_coverage"] == "ALERT":
            recent = (base["day"] > day - pd.Timedelta(days=WINDOW_DAYS)) & (base["day"] <= day)
            offsets[live["version"]] = conformal_offset(v["y"][recent], v["lower"][recent], v["upper"][recent])
            since_calibration = day + pd.Timedelta(days=1)
            actions.append(f"RECALIBRATE (avg offset {offsets[live['version']].mean():+.2f} cars)")
        served["actions"] = "; ".join(actions)

    return pd.DataFrame(log)

## 5. Run A: replay the real forecasts

Both sites, twice: **watch only** (what happens if nobody acts) and **with the policy**.

In [7]:
run_a = {}
for site in SITES:
    v1 = {"v1": model[site]}
    run_a[site] = {"watch": replay(site, v1, act=False), "policy": replay(site, v1, act=True)}
    run_a[site]["policy"].to_csv(f"{OUT}/model_quality_daily_{site}.csv", index=False)


def summarize(res):
    scored = res[res["values"] > 0]
    return {
        "days scored": len(scored),
        "days MAE ratio > 0.80 (7-day)": int((scored["ratio_7d"] > RATIO_SLO).sum()),
        "days coverage outside 75-85%": int((scored["s_coverage"] == "ALERT").sum()),
        "times M2 literal rule fires (3 days > 0.80)": int(res["actions"].str.contains("PAGE").sum()),
        "rollbacks / fallbacks": int(res["actions"].str.contains("ROLLBACK|FALLBACK").sum()),
        "recalibrations": int(res["actions"].str.contains("RECALIBRATE").sum()),
        "overall coverage served": scored["covered"].sum() / scored["values"].sum(),
        "overall MAE served": scored["abs_err"].sum() / scored["values"].sum(),
    }

pd.DataFrame({(site, mode): summarize(r) for site in SITES for mode, r in run_a[site].items()}).round(3)

caltech               jpl  \
                                               watch   policy    watch   
days scored                                  127.000  127.000  131.000   
days MAE ratio > 0.80 (7-day)                 47.000   47.000    3.000   
days coverage outside 75-85%                  26.000    6.000   10.000   
times M2 literal rule fires (3 days > 0.80)    4.000    4.000    0.000   
rollbacks / fallbacks                          0.000    0.000    0.000   
recalibrations                                 0.000    6.000    0.000   
overall coverage served                        0.766    0.799    0.798   
overall MAE served                             0.921    0.921    1.806   

                                                      
                                              policy  
days scored                                  131.000  
days MAE ratio > 0.80 (7-day)                  3.000  
days coverage outside 75-85%                   8.000  
times M2 literal rule fires (3 days > 0.80)    0.000  
rollbacks / fallbacks                          0.000  
recalibrations                                 8.000  
overall coverage served                        0.793  
overall MAE served                             1.806

**What run A shows**

* **The M2 accuracy SLO is too tight for Caltech.** DLinear beats the naive rule by 24% overall, but the rolling
  7-day ratio is above 0.80 on **47 of 127 days**, and the literal M2 rule ("3 days in a row") fires **4 times**.
  The ratio never stays at or above 1.00, so the two-level policy **pages and opens a retraining ticket but never
  rolls back**: the right call, because the fallback would be worse.
* **JPL is comfortably inside the accuracy SLO** (above 0.80 on only 3 of 131 days).
* **Recalibration fixes Caltech's coverage.** Caltech under-covers persistently (76.6%, as in M2). With weekly
  recalibration the served coverage becomes **79.9%** and the days outside 75-85% drop from **26 to 6**, at the
  cost of slightly wider intervals.
* **But it does not help JPL.** JPL already covers 79.8% on average; its weekly swings are noise, and recalibrating
  chases them (79.3% with the policy, 10 → 8 days outside the band). Improvement for the next version:
  recalibrate only when coverage misses the band **two weeks in a row**.
* **Days without forecasts** (JPL, late July) are the data outage found by the data-quality notebook: those
  windows were removed, so the model is not blamed for missing data.

In [8]:
for site in SITES:
    acted = run_a[site]["policy"]
    acted = acted[acted["actions"] != ""]
    print(f"\n{site.upper()}: {len(acted)} days with an action")
    print(acted.assign(day=acted["day"].dt.strftime("%Y-%m-%d"))
          [["day", "ratio_7d", "coverage_7d", "actions"]].round(3).to_string(index=False))


CALTECH: 9 days with an action
       day  ratio_7d  coverage_7d                                                                             actions
2021-05-25     0.743        0.731                                                 RECALIBRATE (avg offset +0.22 cars)
2021-06-01     0.721        0.856                                                 RECALIBRATE (avg offset +0.04 cars)
2021-06-08     0.849        0.738                                                 RECALIBRATE (avg offset +0.24 cars)
2021-06-16     0.847        0.830                                      PAGE: below target 3 days -> retraining ticket
2021-07-21     0.812        0.851 PAGE: below target 3 days -> retraining ticket; RECALIBRATE (avg offset +0.02 cars)
2021-08-11     0.941        0.774                                      PAGE: below target 3 days -> retraining ticket
2021-08-13     0.898        0.748                                                 RECALIBRATE (avg offset +0.20 cars)
2021-08-21     0.840    

## 6. Run B: a broken model release (Caltech)

A realistic bug: version **v2** applies the **per-window normalisation** (Milestone 2, section 2.2) twice. The
model works on occupancy divided by s_i (trailing-day mean + 0.5) and multiplies its output back by s_i; after a
refactor, v2 multiplies by s_i **one extra time**, so busy-hour forecasts are 2-4 times too high. It goes live on
Tuesday 8 June 2021.

Expected: the MAE ratio jumps above 1.00 (worse than the naive rule), after 3 days the policy **rolls back to v1**
automatically, and v1's own calibration comes back with it.

In [9]:
d = np.load("outputs/caltech_occupancy_data.npz")
assert np.array_equal(d["test_origins"], model["caltech"]["origin"])
s_i = d["test_X_hist"][:, -96:].mean(axis=1, keepdims=True) + 0.5          # the model's scale per forecast

v1 = model["caltech"]
v2 = {**v1, "pred": v1["pred"] * s_i, "lower": v1["lower"] * s_i, "upper": v1["upper"] * s_i}

days = naive["caltech"]["day"].unique().sort_values()
release_day = pd.Timestamp("2021-06-08", tz=TZ)
run_b = replay("caltech", {"v1": v1, "v2": v2}, releases={release_day: "v2"})
run_b.to_csv(f"{OUT}/model_quality_daily_caltech_bad_release.csv", index=False)

window = run_b[(run_b["day"] >= release_day - pd.Timedelta(days=2)) & (run_b["day"] <= release_day + pd.Timedelta(days=8))]
window.assign(day=window["day"].dt.strftime("%Y-%m-%d"))[
    ["day", "version", "daily_ratio", "ratio_7d", "s_ratio", "coverage_7d", "actions"]].round(2)

,day,version,daily_ratio,ratio_7d,s_ratio,coverage_7d,actions
27,2021-06-06,v1,0.98,0.76,WARN,NaN,
28,2021-06-07,v1,0.58,0.79,WARN,NaN,
29,2021-06-08,v2,3.51,NaN,n/a,NaN,release v2
30,2021-06-09,v2,2.31,2.80,ALERT,NaN,
31,2021-06-10,v2,3.07,2.89,ALERT,NaN,
32,2021-06-11,v2,2.83,2.87,ALERT,NaN,ROLLBACK v2 -> v1
33,2021-06-12,v1,0.87,NaN,n/a,NaN,
34,2021-06-13,v1,1.06,0.96,ALERT,NaN,
35,2021-06-14,v1,0.67,0.83,ALERT,NaN,
36,2021-06-15,v1,0.91,0.85,ALERT,NaN,PAGE: below target 3 days -> retraining ticket


v2 goes live, its forecasts are far worse than the naive rule (MAE ratio above 1.00), and after 3 days the policy
rolls back to v1 on its own. Without monitoring the bad release would have stayed until someone complained. The
calibration is stored per version, so the rollback restores v1 together with v1's own interval correction.

## 7. Service SLOs: latency and freshness (mock log)

Latency and freshness come from the serving system, not from the model, and the test set has no serving log. We
**simulate** one, clearly labelled as such: one forecast every 15 minutes, latency drawn around the benchmarked
DLinear CPU value (p50 0.155 ms), publication delay around 20 s. Two incidents are injected: a host overload on
day 30 (latency × 400 for 6 hours) and a stalled scheduler on day 60 (forecasts published 3 to 6 minutes late for
8 hours).

In [10]:
rng = np.random.default_rng(42)
ticks = pd.date_range(days[0], days[-1] + pd.Timedelta(hours=23, minutes=45), freq="15min")
service = pd.DataFrame({
    "tick": ticks,
    "latency_ms": rng.lognormal(np.log(0.155), 0.04, len(ticks)),
    "publish_delay_s": rng.normal(20, 5, len(ticks)).clip(5),
})
overload = (service["tick"] >= days[30] + pd.Timedelta(hours=9)) & (service["tick"] < days[30] + pd.Timedelta(hours=15))
stall = (service["tick"] >= days[60] + pd.Timedelta(hours=6)) & (service["tick"] < days[60] + pd.Timedelta(hours=14))
service.loc[overload, "latency_ms"] *= 400
service.loc[stall, "publish_delay_s"] = rng.uniform(180, 360, stall.sum())

daily_service = service.groupby(service["tick"].dt.normalize()).agg(
    p95_latency_ms=("latency_ms", lambda x: np.quantile(x, 0.95)),
    late_forecasts_pct=("publish_delay_s", lambda x: 100 * (x > 120).mean()))
daily_service["latency SLO"] = np.where(daily_service["p95_latency_ms"] > 50, "ALERT", "OK")
daily_service["freshness SLO"] = np.where(daily_service["late_forecasts_pct"] > 0, "ALERT", "OK")
daily_service[(daily_service["latency SLO"] == "ALERT") | (daily_service["freshness SLO"] == "ALERT")].round(2)

,p95_latency_ms,late_forecasts_pct,latency SLO,freshness SLO
tick,,,,
2021-06-09 00:00:00-07:00,64.40,0.00,ALERT,OK
2021-07-09 00:00:00-07:00,0.17,33.33,OK,ALERT


## 8. Weekly summary

This is the dashboard in text form: one row per week, with the numbers the SLOs are about and what the policy did.

* **MAE / naive**: the week's accuracy ratio (target ≤ 0.80; above 1.00 = worse than the naive rule).
* **coverage, no action / with policy**: share of true values inside the 80% interval (target 75-85%).
* **width**: average interval width in cars (recalibration trades width for coverage).
* **negative %**: proxy check, share of predicted values below 0.
* **worst status**: the worst daily status of the week. Each day is judged on the rolling 7 days, so a week can
  show ALERT even when its own ratio looks fine (the window still contained the previous bad days).

In [11]:
SEVERITY = {"n/a": 0, "OK": 1, "WARN": 2, "ALERT": 3}


def weekly_summary(watch, policy):
    def by_week(res):
        monday = res["day"] - pd.to_timedelta(res["day"].dt.dayofweek, unit="D")
        return res.assign(week=monday.dt.strftime("%Y-%m-%d")).groupby("week")
    p, w = by_week(policy), by_week(watch)
    sums = p[["abs_err", "naive_abs_err", "covered", "values", "width", "negative"]].sum()
    table = pd.DataFrame({
        "MAE / naive": (sums["abs_err"] / sums["naive_abs_err"]).replace(np.inf, np.nan),
        "coverage, no action": w["covered"].sum() / w["values"].sum(),
        "coverage, with policy": sums["covered"] / sums["values"],
        "width (cars)": sums["width"] / sums["values"],
        "negative %": 100 * sums["negative"] / sums["values"],
        "worst accuracy status": p["s_ratio"].agg(lambda x: max(x, key=SEVERITY.get)),
        "worst coverage status": p["s_coverage"].agg(lambda x: max(x, key=SEVERITY.get)),
        "versions": p["version"].agg(lambda x: "/".join(dict.fromkeys(x))),
        "actions taken": p["actions"].agg(lambda x: "; ".join(a for a in x if a)),
    })
    return table.round(3)

In [12]:
print("Caltech, run A (real forecasts)\n")
print(weekly_summary(run_a["caltech"]["watch"], run_a["caltech"]["policy"]).to_string())

Caltech, run A (real forecasts)

            MAE / naive  coverage, no action  coverage, with policy  width (cars)  negative % worst accuracy status worst coverage status versions                                                                        actions taken
week                                                                                                                                                                                                                                   
2021-05-10        0.789                0.768                  0.768         2.311      12.385                 ALERT                  WARN       v1                                                                                     
2021-05-17        0.794                0.768                  0.768         2.259      12.686                 ALERT                  WARN       v1                                                                                     
2021-05-24        0.789                

In [13]:
print("JPL, run A (real forecasts)\n")
print(weekly_summary(run_a["jpl"]["watch"], run_a["jpl"]["policy"]).to_string())

JPL, run A (real forecasts)

            MAE / naive  coverage, no action  coverage, with policy  width (cars)  negative % worst accuracy status worst coverage status versions                        actions taken
week                                                                                                                                                                                   
2021-04-19          NaN                0.938                  0.938         2.127      25.000                   n/a                   n/a       v1                                     
2021-04-26        0.654                0.808                  0.808         4.965      14.435                    OK                    OK       v1                                     
2021-05-03        0.689                0.806                  0.806         4.709      16.741                 ALERT                    OK       v1                                     
2021-05-10        0.698                0.791       

In [14]:
print("Caltech, run B (broken release v2 on 2021-06-08). 'No action' = v2 is never rolled back.\n")
watch_b = replay("caltech", {"v1": v1, "v2": v2}, releases={release_day: "v2"}, act=False)
print(weekly_summary(watch_b, run_b).to_string())

Caltech, run B (broken release v2 on 2021-06-08). 'No action' = v2 is never rolled back.

            MAE / naive  coverage, no action  coverage, with policy  width (cars)  negative % worst accuracy status worst coverage status versions                                                                        actions taken
week                                                                                                                                                                                                                                   
2021-05-10        0.789                0.768                  0.768         2.311      12.385                 ALERT                  WARN       v1                                                                                     
2021-05-17        0.794                0.768                  0.768         2.259      12.686                 ALERT                  WARN       v1                                                                    

## 9. Runbook

| Trigger | Who | First steps | Automatic action |
|---|---|---|---|
| Data-quality ALERT (part 1) | data engineering | fix ingestion, backfill | serve labelled fallback, exclude days from SLOs |
| Negative / impossible predictions | ML on-call (ticket) | clip at 0, check the release | none |
| MAE ratio > 0.80 for 3 days | ML on-call (page) | check data alerts, holidays, demand change; open retraining ticket | none |
| MAE ratio ≥ 1.00 for 3 days | ML on-call (page) | find what changed (release, data) | **rollback** to previous version + its calibration, or **fallback** to seasonal naive |
| Coverage outside 75-85% | ML on-call (ticket) | check it recovers after recalibration | **recalibrate** on last 7 days |
| p95 latency > 50 ms | platform (page) | check host load, restart service | serve last cached forecast |
| Forecast > 2 min late | platform (page) | check scheduler | label forecast stale, serve fallback |
| Retraining ticket open > 2 weeks, or PSI vs training > 0.25 (part 1) | ML team | retrain on recent data | new version runs 2 weeks in **shadow mode** before release (M2) |

## 10. Summary

* **Labels arrive 4 hours late**, so the Milestone 2 SLOs can be checked every day on real outcomes.
* **Proxy check:** about 1 in 8 forecasts is negative; clipping at 0 is a free improvement.
* **Accuracy:** on real Caltech data the "MAE ≤ 0.8 × naive" target is often missed, but the model never becomes
  worse than the naive rule, so the two-level policy pages and asks for retraining instead of rolling back to
  something worse.
* **Coverage:** weekly recalibration fixes Caltech's persistent under-coverage (76.6% → 79.9%); on JPL, where
  coverage is already on target, it only chases noise, so it should require two bad weeks in a row.
* **Broken release:** detected and rolled back automatically after 3 days.

Files in `outputs/monitoring/`: `model_quality_daily_<site>.csv` and `model_quality_daily_caltech_bad_release.csv`.